# Midfielders Analysis System Documentation

## Overview
The Midfielders Analysis System is a Python-based tool that evaluates and ranks football/soccer midfielders across multiple competitions and seasons using StatsBomb data. The system calculates various performance metrics to provide a comprehensive assessment of midfielder abilities.

## Core Components

### 1. Performance Metrics

#### Pass Control (0-10 scale)
- Evaluates passing effectiveness with weighted scoring:
  - Short passes (60% weight): Passes under 15 units distance
  - Long passes (40% weight): Passes over 30 units distance
- Completion rates calculated for both categories

#### Ball Progression (0-10 scale)
- Measures midfielder's ability to advance play:
  - Progressive carries (50% weight): Carries advancing ball ≥10 units forward
  - Successful dribbles (50% weight): Ratio of won dribbles to total attempts

#### Defensive Contribution (0-10 scale)
- Assesses defensive capabilities:
  - Tackle success rate (60% weight): Ratio of successful tackles
  - Ball recovery rate (40% weight): Recoveries relative to total actions

#### Creative Output (0-10 scale)
- Evaluates chance creation:
  - Shot assist rate (60% weight): Passes leading to shots
  - Key pass rate (40% weight): Passes leading to goals

### 2. Data Processing

#### Player Identification
- Tracks players in midfielder positions:
  - Defensive midfield (left, center, right)
  - Central midfield (left, center, right)
  - Attacking midfield (left, center, right)
  - Wide midfield (left, right)

#### Data Normalization
- Normalizes metrics considering:
  - Minutes played (minimum threshold: 180 minutes)
  - Performance scaling relative to other players
  - Maximum score cap of 10 points per metric

#### Performance Aggregation
- Combines data across matches and competitions
- Retains best performance period for each player
- Calculates composite score from all metrics

### 3. Analysis Parameters

#### Match Statistics
- Minutes played per match
- Full match equivalents (minutes/90)
- Total matches played
- Team affiliation

#### Competition Coverage
- Multiple leagues and seasons
- Configurable competition and season IDs
- Comprehensive cross-competition analysis

## Output and Results

### Data Export
- Saves complete analysis to CSV file
- Filename: "consolidated_midfielder_analysis.csv"

### Key Display Metrics
- Player name and team
- Matches played
- Composite performance score
- Individual metric scores:
  - Pass control
  - Ball progression
  - Defensive contribution
  - Creative output

## Technical Implementation

### Dependencies
- pandas: Data manipulation and analysis
- numpy: Numerical computations
- statsbombpy: Data source integration
- collections: Data structure utilities

### Error Handling
- Robust exception management for:
  - Data retrieval issues
  - Calculation errors
  - Missing or incomplete data

### Performance Optimization
- Efficient data structures using defaultdict
- Streamlined metric calculations
- Selective data retention for memory efficiency

## Usage

### Configuration
```python
leagues = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]},
    # Additional leagues and seasons...
]
```

### Execution
```python
midfielder_analysis = analyze_multiple_competitions(leagues)
```

### Output Review
- System displays top 10 midfielders by composite score
- Complete dataset saved to CSV for further analysis

## Data Flow

1. Competition and season data retrieval
2. Event-based performance tracking
3. Metric calculation and normalization
4. Player performance aggregation
5. Results compilation and ranking
6. Data export and display

This system provides a comprehensive evaluation of midfielder performance, enabling objective comparison across different competitions and seasons while accounting for various aspects of midfielder play.

In [ ]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from collections import defaultdict
import warnings
warnings.simplefilter("ignore")

# [Previous metric calculation functions remain the same]
def calculate_pass_control(events, midfielder, team):
    """Calculate passing metrics including short and long passing effectiveness"""
    midfielder_events = events[events['player'] == midfielder]
    
    passes = midfielder_events[midfielder_events['type'] == 'Pass']
    if len(passes) == 0:
        return 0
        
    def is_short_pass(pass_row):
        if not (isinstance(pass_row['location'], list) and isinstance(pass_row['pass_end_location'], list)):
            return False
        start = pass_row['location']
        end = pass_row['pass_end_location']
        distance = np.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
        return distance < 15
        
    short_passes = passes[passes.apply(is_short_pass, axis=1)]
    short_completion = (
        len(short_passes[short_passes['pass_outcome'].isna()]) /
        len(short_passes) if len(short_passes) > 0 else 0
    )
    
    def is_long_pass(pass_row):
        if not (isinstance(pass_row['location'], list) and isinstance(pass_row['pass_end_location'], list)):
            return False
        start = pass_row['location']
        end = pass_row['pass_end_location']
        distance = np.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
        return distance > 30
        
    long_passes = passes[passes.apply(is_long_pass, axis=1)]
    long_completion = (
        len(long_passes[long_passes['pass_outcome'].isna()]) /
        len(long_passes) if len(long_passes) > 0 else 0
    )
    
    return (short_completion * 0.6) + (long_completion * 0.4)

def calculate_ball_progression(events, midfielder, team):
    """Calculate ball progression metrics including carries and dribbles"""
    midfielder_events = events[events['player'] == midfielder]
    
    carries = midfielder_events[midfielder_events['type'] == 'Carry']
    if len(carries) == 0:
        return 0
        
    def is_progressive_carry(carry_row):
        if not (isinstance(carry_row['location'], list) and isinstance(carry_row['carry_end_location'], list)):
            return False
        start = carry_row['location']
        end = carry_row['carry_end_location']
        return end[0] - start[0] >= 10
        
    progressive_carries = carries[carries.apply(is_progressive_carry, axis=1)]
    carry_score = len(progressive_carries) / len(carries)
    
    dribbles = midfielder_events[
        (midfielder_events['type'] == 'Duel') &
        (midfielder_events['duel_type'] == 'Dribble')
    ]
    
    dribble_success = (
        len(dribbles[dribbles['duel_outcome'] == 'Won']) /
        len(dribbles) if len(dribbles) > 0 else 0
    )
    
    return (carry_score * 0.5) + (dribble_success * 0.5)

def calculate_defensive_contribution(events, midfielder, team):
    """Calculate defensive metrics including tackles and recoveries"""
    midfielder_events = events[events['player'] == midfielder]
    
    tackles = midfielder_events[
        (midfielder_events['type'] == 'Duel') &
        (midfielder_events['duel_type'] == 'Tackle')
    ]
    
    tackle_success = (
        len(tackles[tackles['duel_outcome'] == 'Won']) /
        len(tackles) if len(tackles) > 0 else 0
    )
    
    recoveries = midfielder_events[midfielder_events['type'] == 'Ball Recovery']
    recovery_rate = len(recoveries) / max(len(midfielder_events), 1)
    
    return (tackle_success * 0.6) + (recovery_rate * 0.4)

def calculate_creative_output(events, midfielder, team):
    """Calculate creative metrics including chance creation and pre-assists"""
    try:
        midfielder_events = events[events['player'] == midfielder]
        
        passes = midfielder_events[midfielder_events['type'] == 'Pass']
        if len(passes) == 0:
            return 0
        
        if 'pass_shot_assist' in passes.columns:
            shot_assists = passes[passes['pass_shot_assist'] == True]
            shot_assist_rate = len(shot_assists) / len(passes)
        else:
            if 'shot_assist' in passes.columns:
                shot_assists = passes[passes['shot_assist'] == True]
                shot_assist_rate = len(shot_assists) / len(passes)
            else:
                shot_assist_rate = 0
        
        if 'pass_goal_assist' in passes.columns:
            key_passes = passes[passes['pass_goal_assist'] == True]
            key_pass_rate = len(key_passes) / len(passes)
        else:
            if 'goal_assist' in passes.columns:
                key_passes = passes[passes['goal_assist'] == True]
                key_pass_rate = len(key_passes) / len(passes)
            else:
                key_pass_rate = 0
        
        return (shot_assist_rate * 0.6) + (key_pass_rate * 0.4)
        
    except Exception as e:
        print(f"Warning: Error calculating creative output for {midfielder}: {str(e)}")
        return 0

def normalize_metric(series, min_val=0, max_val=10, minutes_played=None):
    """Normalize metric with minutes adjustment"""
    if len(series) == 0:
        return pd.Series([])
        
    if minutes_played is not None:
        min_minutes = 180
        weights = np.minimum(np.maximum(minutes_played, min_minutes) / 270, 1)
        series = series * weights
    
    percentiles = series.rank(pct=True)
    normalized = percentiles * max_val
    normalized = normalized.clip(min_val, max_val)
    
    return normalized.round(2)

def analyze_multiple_competitions(leagues):
    """
    Analyze midfielder performance across multiple competitions and seasons
    Returns a consolidated DataFrame with the best performance for each player
    """
    all_midfielder_data = []
    
    midfielder_positions = [
        'Right Defensive Midfield',
        'Center Defensive Midfield',
        'Left Defensive Midfield',
        'Left Center Midfield',
        'Right Center Midfield',
        'Right Midfield',
        'Center Midfield',
        'Left Midfield',
        'Right Attacking Midfield',
        'Center Attacking Midfield',
        'Left Attacking Midfield'
    ]
    
    for league in leagues:
        competition_id = league['competition_id']
        for season_id in league['season_ids']:
            print(f"\nAnalyzing competition {competition_id}, season {season_id}...")
            try:
                matches = sb.matches(competition_id=competition_id, season_id=season_id)
                
                midfielder_stats = defaultdict(lambda: {
                    'minutes_played': 0,
                    'pass_control': 0,
                    'ball_progression': 0,
                    'defensive_contribution': 0,
                    'creative_output': 0,
                    'matches_played': 0,
                    'team': None,
                    'competition_id': competition_id,
                    'season_id': season_id
                })
                
                for _, match in matches.iterrows():
                    events = sb.events(match_id=match['match_id'])
                    
                    midfielders = events[
                        events['position'].isin(midfielder_positions)
                    ]['player'].unique()
                    
                    for midfielder in midfielders:
                        midfielder_events = events[events['player'] == midfielder]
                        if len(midfielder_events) == 0:
                            continue
                        
                        team = midfielder_events['team'].iloc[0]
                        minutes = midfielder_events['minute'].max() - midfielder_events['minute'].min()
                        
                        stats = midfielder_stats[midfielder]
                        stats['minutes_played'] += minutes
                        stats['pass_control'] += calculate_pass_control(events, midfielder, team)
                        stats['ball_progression'] += calculate_ball_progression(events, midfielder, team)
                        stats['defensive_contribution'] += calculate_defensive_contribution(events, midfielder, team)
                        stats['creative_output'] += calculate_creative_output(events, midfielder, team)
                        stats['matches_played'] += 1
                        stats['team'] = team
                
                df = pd.DataFrame.from_dict(midfielder_stats, orient='index')
                if len(df) > 0:
                    all_midfielder_data.append(df)
                    
            except Exception as e:
                print(f"Error analyzing competition {competition_id}, season {season_id}: {str(e)}")
                continue
    
    if not all_midfielder_data:
        print("No data collected!")
        return pd.DataFrame()
    
    # Combine all data
    combined_df = pd.concat(all_midfielder_data, ignore_index=False)
    
    # Reset index to make player names a column
    combined_df = combined_df.reset_index().rename(columns={'index': 'midfielder'})
    
    # Keep only the entry with most minutes for each player
    combined_df = combined_df.sort_values('minutes_played', ascending=False)
    combined_df = combined_df.drop_duplicates(subset=['midfielder'], keep='first')
    
    # Normalize metrics
    metrics = ['pass_control', 'ball_progression', 'defensive_contribution', 'creative_output']
    for metric in metrics:
        combined_df[metric] = combined_df[metric] / combined_df['matches_played']
        combined_df[metric] = normalize_metric(
            combined_df[metric],
            minutes_played=combined_df['minutes_played']
        )
    
    # Calculate additional metrics
    combined_df['avg_minutes_per_match'] = (combined_df['minutes_played'] / combined_df['matches_played']).round(1)
    combined_df['full_match_equivalent'] = (combined_df['minutes_played'] / 90).round(1)
    combined_df['composite_score'] = combined_df[metrics].mean(axis=1)
    
    # Sort by composite score
    combined_df = combined_df.sort_values('composite_score', ascending=False)
    
    return combined_df

# Main execution
leagues = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]},
    {"competition_id": 11, "season_ids": [90, 42, 4]},
    {"competition_id": 7, "season_ids": [235, 108]},
    {"competition_id": 2, "season_ids": [44]},
    {"competition_id": 12, "season_ids": [27]},
    {"competition_id": 55, "season_ids": [282]},
]

# Run analysis
print("Starting midfielder analysis across multiple competitions...")
midfielder_analysis = analyze_multiple_competitions(leagues)

# Save results
if len(midfielder_analysis) > 0:
    output_filename = "consolidated_midfielder_analysis.csv"
    midfielder_analysis.to_csv(output_filename, index=False)
    print(f"\nResults saved to {output_filename}")
    
    # Display top 10 performers
    print("\nTop 10 Midfielders by Composite Score:")
    print("-" * 80)
    display_columns = [
        'midfielder', 'team', 'matches_played', 
        'composite_score', 'pass_control', 'ball_progression', 
        'defensive_contribution', 'creative_output'
    ]
    print(midfielder_analysis[display_columns].head(10))
else:
    print("No results to save!")